In [36]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
import anndata
from scipy.stats import zscore
import pickle
import tqdm
from datasets import Dataset, Features, Sequence, Value
import os

In [66]:
from genecompass import BertForMaskedLM

In [45]:
# ============ 配置路径 ============
BASE_PATH = '/media/ubuntu/sda/GeneCompass-main'

# 输入文件路径
INPUT_PATH = f'{BASE_PATH}/allen_brain_atlas/WMB-10Xv3-CB-raw.h5ad'

# 字典文件路径
# 注意：目前只有human的中值字典。处理mouse数据时：
# - 如果mouse基因在human字典中，会使用human的中值
# - 如果mouse基因不在human字典中，Normalized函数会使用默认值1
# - 这与pretrain_test.py的处理方式一致
DICT_PATH = f'{BASE_PATH}/scdata/dict/human_gene_median_after_filter.pickle'  # 中值字典路径
GENE_TOKEN_PATH = f'{BASE_PATH}/prior_knowledge/h&m_token1000W.pickle'  # token路径（human和mouse共用，直接使用此字典）
GENE_ID_PATH = f'{BASE_PATH}/scdata/gene_id_hpromoter.pickle'  # 基因ID列表（用于过滤）

# 基因列表文件路径
F_LIST = [
    f'{BASE_PATH}/scdata/mouse_protein_coding.txt',
    f'{BASE_PATH}/scdata/human_protein_coding.txt',
    f'{BASE_PATH}/scdata/mouse_miRNA.txt',
    f'{BASE_PATH}/scdata/human_miRNA.txt',
    f'{BASE_PATH}/scdata/human_mitochondria.xlsx',
    f'{BASE_PATH}/scdata/mouse_mitochondria.xlsx'
]

# 物种设置 ('human' 或 'mouse')
# 注意：species_label对应关系：0=human, 1=mouse（与pretrain_test.py一致）
SPECIES_STR = 'mouse'  # 根据Allen Brain Atlas数据设置为mouse

print("配置路径加载完成！")
print(f"当前处理物种: {SPECIES_STR}")

配置路径加载完成！
当前处理物种: mouse


In [55]:

with open(GENE_TOKEN_PATH, 'rb') as f:
    dict1 = pickle.load(f)

In [46]:
# ============ 读取h5ad数据 ============
print(f"正在读取h5ad文件: {INPUT_PATH}")
adata = sc.read_h5ad(INPUT_PATH)
print(f"原始数据形状: {adata.shape}")
print(f"细胞数量: {adata.n_obs}, 基因数量: {adata.n_vars}")

正在读取h5ad文件: /media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/WMB-10Xv3-CB-raw.h5ad
原始数据形状: (182026, 32285)
细胞数量: 182026, 基因数量: 32285


In [47]:
# ============ 保存表达矩阵为CSV格式（用于preprocess.py输入） ============
# 这个cell可以将表达矩阵保存为CSV格式，方便后续使用preprocess.py处理

# 配置：要保存的细胞数量
NUM_CELLS_TO_SAVE = 1000  # 改为你需要的数量，如果为None则保存全部细胞

# 输出文件路径
CSV_OUTPUT_PATH = f'{BASE_PATH}/allen_brain_atlas/allen_brain_atlas_{NUM_CELLS_TO_SAVE if NUM_CELLS_TO_SAVE else "all"}_cells.csv'

print(f"\n============ 保存表达矩阵为CSV格式 ============")
print(f"将保存到: {CSV_OUTPUT_PATH}")

# 优先使用原始数据（adata_raw_copy），如果不存在则使用当前adata
# 注意：原始数据更适合用于preprocess.py，因为preprocess.py会进行标准化和log转换

try:
    # 优先使用原始数据
    if 'adata_raw_copy' in locals():
        print("\n✓ 使用原始数据（adata_raw_copy）- 推荐用于preprocess.py")
        data_source = adata_raw_copy
    elif 'adata' in locals():
        print("\n⚠ 使用当前adata（可能是处理后的数据）")
        print("  提示：如果在预处理之后运行，建议使用原始数据")
        data_source = adata
    else:
        print("错误：adata变量不存在，请先运行前面的cell读取h5ad数据")
        data_source = None
    
    if data_source is not None:
        print(f"\n数据源信息:")
        print(f"  - 形状: {data_source.shape}")
        print(f"  - 细胞数量: {data_source.n_obs}")
        print(f"  - 基因数量: {data_source.n_vars}")
        
        # 选择要保存的细胞数量
        if NUM_CELLS_TO_SAVE is not None and NUM_CELLS_TO_SAVE < data_source.n_obs:
            print(f"\n选择前 {NUM_CELLS_TO_SAVE} 个细胞进行保存...")
            adata_to_save = data_source[:NUM_CELLS_TO_SAVE, :].copy()
        else:
            print(f"\n保存全部 {data_source.n_obs} 个细胞...")
            adata_to_save = data_source.copy()
        
        # 准备CSV格式：第一列是基因ID，其他列是细胞的表达数据
        # 注意：需要转置，使得行是基因，列是细胞
        print("\n正在转换数据格式...")
        
        # 处理稀疏矩阵
        if sparse.issparse(adata_to_save.X):
            expression_matrix = adata_to_save.X.toarray().T
        else:
            expression_matrix = np.asarray(adata_to_save.X).T
        
        # 创建DataFrame
        # 行：基因ID（adata.var.index）
        # 列：细胞ID（adata.obs.index）
        df_output = pd.DataFrame(
            expression_matrix,
            index=adata_to_save.var.index,  # 基因ID作为行索引
            columns=adata_to_save.obs.index  # 细胞ID作为列名
        )
        
        # 重置索引，使得基因ID成为第一列
        df_output.reset_index(inplace=True)
        df_output.rename(columns={'index': 'Unnamed: 0'}, inplace=True)
        
        # 保存为CSV
        print(f"正在保存到: {CSV_OUTPUT_PATH}")
        df_output.to_csv(CSV_OUTPUT_PATH, index=False)
        
        print(f"\n✓ CSV文件保存成功！")
        print(f"  - 文件路径: {CSV_OUTPUT_PATH}")
        print(f"  - CSV形状: {df_output.shape}")
        print(f"  - 基因数量: {len(df_output)}")
        print(f"  - 细胞数量: {len(df_output.columns) - 1}")  # 减去第一列基因列
        print(f"\n这个CSV文件可以直接用于preprocess.py处理！")
        print(f"使用方法：修改preprocess.py第256行为：")
        print(f"  df = pd.read_csv('{CSV_OUTPUT_PATH}')")
        
except Exception as e:
    print(f"错误：{str(e)}")
    import traceback
    traceback.print_exc()


============ 保存表达矩阵为CSV格式 ============
将保存到: /media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/allen_brain_atlas_1000_cells.csv

⚠ 使用当前adata（可能是处理后的数据）
  提示：如果在预处理之后运行，建议使用原始数据

数据源信息:
  - 形状: (182026, 32285)
  - 细胞数量: 182026
  - 基因数量: 32285

选择前 1000 个细胞进行保存...

正在转换数据格式...
正在保存到: /media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/allen_brain_atlas_1000_cells.csv

✓ CSV文件保存成功！
  - 文件路径: /media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/allen_brain_atlas_1000_cells.csv
  - CSV形状: (32285, 1001)
  - 基因数量: 32285
  - 细胞数量: 1000

这个CSV文件可以直接用于preprocess.py处理！
使用方法：修改preprocess.py第256行为：
  df = pd.read_csv('/media/ubuntu/sda/GeneCompass-main/allen_brain_atlas/allen_brain_atlas_1000_cells.csv')


In [67]:
with open('/media/ubuntu/sda/GeneCompass-main/prior_knowledge/human_mouse_tokens.pickle', 'rb') as f:
    dict1 = pickle.load(f)

In [68]:
symbol2ENSMUSG_dict = {}
ENSMUSG2symbol_dict = {}
for i in range(len(adata.var)):
    if adata.var.index[i] in (dict1.keys()):
        symbol2ENSMUSG_dict[adata.var['gene_symbol'].values[i]] = adata.var.index[i]
        ENSMUSG2symbol_dict[adata.var.index[i]] = adata.var['gene_symbol'].values[i]

In [69]:
with open("/media/ubuntu/sda/GeneCompass-main/scdata/symbol2ENSMUG_dict.pickle", 'wb') as f:
    pickle.dump(symbol2ENSMUSG_dict, f)

with open("/media/ubuntu/sda/GeneCompass-main/scdata/ENSMUSG2symbol_dict.pickle", 'wb') as f:
    pickle.dump(ENSMUSG2symbol_dict, f)